# AI Refund Approval — Block [3] & [4]: Feature Extraction + Classification

This notebook applies **Block [3] BERT Feature Extraction** and **Block [4] Classification**
on all complaints that passed the primitive validation pipeline.

**Input**:  `validation_results.csv`  (VALID decisions only)  +  `model.pkl`
**Output**: `predictions.csv`  (APPROVE / REJECT per complaint)

```
VALID Complaints
      ↓
[3] Feature Extraction
    ├── Sentence-BERT embedding (384-dim, all-MiniLM-L6-v2)
    ├── Issue Type  (Sentence-BERT cosine similarity → 5 categories)
    ├── Severity    (Sentence-BERT cosine similarity → High/Med/Low)
    ├── Sentiment   (TextBlob polarity)
    └── Urgency     (keyword detection)
      ↓
[4] XGBoost Classifier
    └── APPROVE (high severity) / REJECT (medium or low severity)
```


## 1. Imports & Configuration

In [1]:
import re
import sys
import numpy as np
import pandas as pd
import joblib
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from textblob import TextBlob
import warnings
warnings.filterwarnings("ignore")

# ── Paths ──
VALIDATION_CSV  = "validation_results.csv"
MODEL_PKL       = "model.pkl"
PREDICTIONS_CSV = "predictions.csv"

# ── Sentence-BERT model ──
MODEL_NAME = "all-MiniLM-L6-v2"
print("Loading Sentence-BERT model...")
bert_model = SentenceTransformer(MODEL_NAME)
print("Model loaded.\n")

# ── Embedding cache ──
_emb_cache: dict = {}

def _embed(text: str) -> np.ndarray:
    """Return cached Sentence-BERT embedding."""
    if text not in _emb_cache:
        _emb_cache[text] = bert_model.encode(text, show_progress_bar=False)
    return _emb_cache[text]

def _max_sim(text_emb: np.ndarray, refs: list) -> tuple:
    ref_embs = np.array([_embed(r) for r in refs])
    sims = cosine_similarity(text_emb.reshape(1, -1), ref_embs)[0]
    idx = int(np.argmax(sims))
    return float(sims[idx]), idx

print("Configuration ready.")


Loading Sentence-BERT model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded.

Configuration ready.


## 2. Reference Phrases (Sentence-BERT)

All semantic similarity checks use the **same Sentence-BERT model** (`all-MiniLM-L6-v2`).
- **Issue type**: cosine similarity vs 5 category phrases
- **Severity**: cosine similarity vs High / Medium / Low tier phrases


In [2]:
# ── Reference phrases (Sentence-BERT used for ALL similarity tasks) ──

ISSUE_TYPE_REFS = [
    "screen damage cracked display broken screen",   # 0
    "battery draining not charging battery problem",  # 1
    "performance slow lagging freezing",              # 2
    "connectivity disconnecting wifi bluetooth",      # 3
    "hardware failure dead not working broken",       # 4
]

ISSUE_TYPE_NAMES = ["screen", "battery", "performance", "connectivity", "hardware"]

SEVERITY_HIGH_REFS = [
    "completely broken and unusable",
    "dead on arrival will not turn on",
    "cracked screen device destroyed",
    "not working at all hardware failure",
    "completely non functional",
]

SEVERITY_MEDIUM_REFS = [
    "disconnecting frequently from bluetooth",
    "overheating during normal use",
    "randomly restarting several times a day",
    "battery drains very fast",
    "camera blurry all the time",
]

SEVERITY_LOW_REFS = [
    "slightly slow minor delay",
    "small cosmetic scratch on the body",
    "very minor issue barely noticeable",
    "slight audio delay not a big problem",
    "minor creak in chassis",
]

URGENCY_KEYWORDS = {"asap", "urgent", "urgently", "immediately",
                    "right away", "as soon as possible"}

# Pre-warm cache
print("Pre-warming Sentence-BERT cache with reference phrases...")
all_refs = (ISSUE_TYPE_REFS + SEVERITY_HIGH_REFS
            + SEVERITY_MEDIUM_REFS + SEVERITY_LOW_REFS)
for ref in all_refs:
    _embed(ref)
print(f"Cache warmed — {len(_emb_cache)} entries.\n")


Pre-warming Sentence-BERT cache with reference phrases...


Cache warmed — 20 entries.



## 3. `preprocess()` — Text Normalisation

Minimal cleaning to match the pre-processing used in Block [1] / [2].


In [3]:
def preprocess(text: str) -> str:
    """
    Minimal normalisation:
    - Lowercase
    - Replace 5+ digit numbers with <ID>
    - Normalise repeated characters (3+ → 2)
    - Strip non-essential symbols
    - Collapse whitespace
    """
    text = text.lower()
    text = re.sub(r"\b\d{5,}\b", "<ID>", text)
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)
    text = re.sub(r"[^\w\s!?.,]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# ── Quick test ──
tests = [
    "The iPhone 14 screen is CRACKED. All caps urgency!",
    "order #1234567890 is bad",
    "soooo frustrating",
]
for t in tests:
    print(f"  {t!r}")
    print(f"  → {preprocess(t)!r}\n")


  'The iPhone 14 screen is CRACKED. All caps urgency!'
  → 'the iphone 14 screen is cracked. all caps urgency!'

  'order #1234567890 is bad'
  → 'order ID is bad'

  'soooo frustrating'
  → 'soo frustrating'



## 4. `extract_features()` — Feature Extraction (Block [3])

Produces a **388-dimensional feature vector** per complaint:
| Dims | Feature | Method |
|------|---------|--------|
| 0–383 | BERT embedding | Sentence-BERT `all-MiniLM-L6-v2` |
| 384 | Severity (1/2/3) | Sentence-BERT cosine similarity |
| 385 | Sentiment | TextBlob polarity |
| 386 | Urgency | Keyword match |
| 387 | Issue type (0-4) | Sentence-BERT cosine similarity |


In [4]:
def extract_features(text: str) -> dict:
    """
    Extract all five features from complaint text using Sentence-BERT.

    Feature vector layout (388 dims):
      [0:384]  Sentence-BERT embedding
      [384]    severity  (1=Low, 2=Medium, 3=High)
      [385]    sentiment (-1.0 to +1.0, TextBlob polarity)
      [386]    urgency   (0 or 1, keyword-based)
      [387]    issue_type (0-4, Sentence-BERT cosine similarity)
    """
    cleaned = preprocess(text)
    emb = _embed(cleaned)

    # Issue Type — Sentence-BERT cosine similarity
    _, issue_idx = _max_sim(emb, ISSUE_TYPE_REFS)

    # Severity — Sentence-BERT cosine similarity to 3-tier references
    high_score,   _ = _max_sim(emb, SEVERITY_HIGH_REFS)
    medium_score, _ = _max_sim(emb, SEVERITY_MEDIUM_REFS)
    low_score,    _ = _max_sim(emb, SEVERITY_LOW_REFS)

    scores = {3: high_score, 2: medium_score, 1: low_score}
    severity = max(scores, key=scores.get)

    # Sentiment — TextBlob polarity
    sentiment = round(TextBlob(cleaned).sentiment.polarity, 4)

    # Urgency — keyword match
    words = set(cleaned.split())
    urgency = int(bool(words & URGENCY_KEYWORDS))

    # Combined feature vector
    feature_vector = (
        list(emb.astype(float))
        + [float(severity)]
        + [float(sentiment)]
        + [float(urgency)]
        + [float(issue_idx)]
    )

    return {
        "cleaned_text":   cleaned,
        "embedding":      emb,
        "issue_type":     issue_idx,
        "issue_name":     ISSUE_TYPE_NAMES[issue_idx],
        "severity":       severity,
        "sentiment":      sentiment,
        "urgency":        urgency,
        "feature_vector": feature_vector,
    }

print("extract_features() function defined.")

# ── Smoke test ──
test_text = "The iPhone 14 screen is completely cracked and won't turn on. Need help ASAP."
r = extract_features(test_text)
print(f"\nSmoke test:")
print(f"  Text      : {test_text}")
print(f"  Issue     : {r['issue_name']}  (idx={r['issue_type']})")
print(f"  Severity  : {r['severity']}  ({'High' if r['severity']==3 else 'Medium' if r['severity']==2 else 'Low'})")
print(f"  Sentiment : {r['sentiment']}")
print(f"  Urgency   : {r['urgency']}")
print(f"  Vec dims  : {len(r['feature_vector'])}")


extract_features() function defined.



Smoke test:
  Text      : The iPhone 14 screen is completely cracked and won't turn on. Need help ASAP.
  Issue     : screen  (idx=0)
  Severity  : 3  (High)
  Sentiment : 0.1
  Urgency   : 0
  Vec dims  : 388


## 5. `classify()` — Inference Function (Block [4])

Wraps the trained XGBoost model to output `APPROVE` or `REJECT` for any text.


In [5]:
def classify(text: str, model) -> str:
    """
    Run inference for a single complaint.
    Returns "APPROVE" or "REJECT".
    """
    feats = extract_features(text)
    vec   = np.array(feats["feature_vector"]).reshape(1, -1)
    pred  = int(model.predict(vec)[0])
    return "APPROVE" if pred == 1 else "REJECT"

print("classify() function defined.")


classify() function defined.


## 6. Load Validated Complaints

In [6]:
# ── Load validation results (VALID only) ──
print(f"Loading '{VALIDATION_CSV}'...")
df_all = pd.read_csv(VALIDATION_CSV)
df = df_all[df_all["decision"] == "VALID"].copy().reset_index(drop=True)
print(f"Total rows    : {len(df_all)}")
print(f"VALID rows    : {len(df)}"  )
print(df[["complaint_id", "complaint_text"]].head(5))


Loading 'validation_results.csv'...
Total rows    : 225
VALID rows    : 160
   complaint_id                                     complaint_text
0             1  Got the Apple iPhone 14 and the screen was cra...
1             2  My JBL Tune 760NC Headphones is completely bro...
2             3  Got the Apple AirPods Pro 2 and the screen was...
3             4  The Apple iPhone 14 stopped working after 2 da...
4             5  My MX Master 3S Mouse broke within one week of...


## 7. Load Trained Model (`model.pkl`)

In [7]:
# ── Load trained model ──
print(f"\nLoading model from '{MODEL_PKL}'...")
clf = joblib.load(MODEL_PKL)
print(f"Model loaded  : {type(clf).__name__}")



Loading model from 'model.pkl'...


Model loaded  : XGBClassifier

## 8. Run Inference on All VALID Complaints

In [8]:
# ── Run inference on all VALID complaints ──
print(f"\nRunning feature extraction + classification on {len(df)} complaints...\n")

results = []
for i, (_, row) in enumerate(df.iterrows(), 1):
    feats = extract_features(row["complaint_text"])
    pred  = classify(row["complaint_text"], clf)

    results.append({
        "complaint_id":  row["complaint_id"],
        "complaint_text": row["complaint_text"],
        "cleaned_text":  feats["cleaned_text"],
        "issue_type":    feats["issue_name"],
        "severity":      feats["severity"],
        "severity_label": ("High" if feats["severity"] == 3
                           else "Medium" if feats["severity"] == 2 else "Low"),
        "sentiment":     feats["sentiment"],
        "urgency":       feats["urgency"],
        "prediction":    pred,
    })

    if i % 30 == 0 or i == len(df):
        print(f"  Processed {i}/{len(df)}...")

preds_df = pd.DataFrame(results)
print(f"\nDone. {len(preds_df)} predictions generated.")



Running feature extraction + classification on 160 complaints...



  Processed 30/160...


  Processed 60/160...


  Processed 90/160...


  Processed 120/160...


  Processed 150/160...


  Processed 160/160...

Done. 160 predictions generated.


## 9. Results Summary

In [9]:
# ── Summary ──
total    = len(preds_df)
approved = (preds_df["prediction"] == "APPROVE").sum()
rejected = (preds_df["prediction"] == "REJECT").sum()

print("=" * 60)
print("        CLASSIFICATION RESULTS SUMMARY")
print("=" * 60)
print(f"  Total VALID complaints : {total}")
print(f"  APPROVE                : {approved}  ({approved/total*100:.1f}%)")
print(f"  REJECT                 : {rejected}  ({rejected/total*100:.1f}%)")
print()
print("Severity breakdown (VALID complaints):")
sev_map = {3: "High", 2: "Medium", 1: "Low"}
for sev_val, sev_name in sev_map.items():
    n = (preds_df["severity"] == sev_val).sum()
    app = ((preds_df["severity"] == sev_val) & (preds_df["prediction"] == "APPROVE")).sum()
    rej = n - app
    print(f"  {sev_name:6s} (sev={sev_val}) : {n:3d} total | APPROVE={app} | REJECT={rej}")

print()
print("Issue type breakdown:")
for itype, count in preds_df["issue_type"].value_counts().items():
    app = ((preds_df["issue_type"] == itype) & (preds_df["prediction"] == "APPROVE")).sum()
    print(f"  {itype:15s}: {count:3d} complaints  |  APPROVE={app}")


        CLASSIFICATION RESULTS SUMMARY
  Total VALID complaints : 160
  APPROVE                : 77  (48.1%)
  REJECT                 : 83  (51.9%)

Severity breakdown (VALID complaints):
  High   (sev=3) :  77 total | APPROVE=77 | REJECT=0
  Medium (sev=2) :  48 total | APPROVE=0 | REJECT=48
  Low    (sev=1) :  35 total | APPROVE=0 | REJECT=35

Issue type breakdown:
  hardware       :  61 complaints  |  APPROVE=44
  screen         :  37 complaints  |  APPROVE=30
  performance    :  30 complaints  |  APPROVE=0
  connectivity   :  19 complaints  |  APPROVE=0
  battery        :  13 complaints  |  APPROVE=3


## 10. Save `predictions.csv` + Sample Output

In [10]:
# ── Save predictions.csv ──
preds_df.to_csv(PREDICTIONS_CSV, index=False)
print(f"\nSaved '{PREDICTIONS_CSV}' with {len(preds_df)} rows.\n")

# ── Sample APPROVE ──
print("=" * 60)
print("  SAMPLE APPROVED COMPLAINTS")
print("=" * 60)
approved_samples = preds_df[preds_df["prediction"] == "APPROVE"].head(6)
for _, r in approved_samples.iterrows():
    print(f"\n  ID={r['complaint_id']:3d} | Severity={r['severity_label']:6s} | Issue={r['issue_type']}")
    print(f"  Text: {r['complaint_text'][:95]}")

# ── Sample REJECT ──
print("\n" + "=" * 60)
print("  SAMPLE REJECTED COMPLAINTS")
print("=" * 60)
rejected_samples = preds_df[preds_df["prediction"] == "REJECT"].head(6)
for _, r in rejected_samples.iterrows():
    print(f"\n  ID={r['complaint_id']:3d} | Severity={r['severity_label']:6s} | Issue={r['issue_type']}")
    print(f"  Text: {r['complaint_text'][:95]}")



Saved 'predictions.csv' with 160 rows.

  SAMPLE APPROVED COMPLAINTS

  ID=  1 | Severity=High   | Issue=screen
  Text: Got the Apple iPhone 14 and the screen was cracked when I opened the package.

  ID=  2 | Severity=High   | Issue=screen
  Text: My JBL Tune 760NC Headphones is completely broken. It fell once and now the screen is cracked b

  ID=  3 | Severity=High   | Issue=screen
  Text: Got the Apple AirPods Pro 2 and the screen was cracked when I opened the package.

  ID=  4 | Severity=High   | Issue=hardware
  Text: The Apple iPhone 14 stopped working after 2 days. Completely dead, not charging.

  ID=  5 | Severity=High   | Issue=hardware
  Text: My MX Master 3S Mouse broke within one week of purchase. It won't turn on anymore.

  ID=  6 | Severity=High   | Issue=screen
  Text: Got the Logitech MX Master 3S Mouse and the screen was cracked when I opened the package.

  SAMPLE REJECTED COMPLAINTS

  ID= 46 | Severity=Low    | Issue=performance
  Text: The Logitech MX Master 3

## 11. Full Predictions Table (Styled)

In [11]:
# ── Styled results table ──
preds_df.style.map(
    lambda v: "background-color:#d4edda" if v == "APPROVE" else
              "background-color:#f8d7da" if v == "REJECT"  else "",
    subset=["prediction"]
).map(
    lambda v: "background-color:#fff3cd" if v == "High" else
              "background-color:#d1ecf1" if v == "Medium" else "",
    subset=["severity_label"]
)


,complaint_id,complaint_text,cleaned_text,issue_type,severity,severity_label,sentiment,urgency,prediction
0,1,Got the Apple iPhone 14 and the screen was cracked when I opened the package.,got the apple iphone 14 and the screen was cracked when i opened the package.,screen,3,High,0.000000,0,APPROVE
1,2,My JBL Tune 760NC Headphones is completely broken. It fell once and now the screen is cracked beyond repair.,my jbl tune 760nc headphones is completely broken. it fell once and now the screen is cracked beyond repair.,screen,3,High,-0.400000,0,APPROVE
2,3,Got the Apple AirPods Pro 2 and the screen was cracked when I opened the package.,got the apple airpods pro 2 and the screen was cracked when i opened the package.,screen,3,High,0.000000,0,APPROVE
3,4,"The Apple iPhone 14 stopped working after 2 days. Completely dead, not charging.","the apple iphone 14 stopped working after 2 days. completely dead, not charging.",hardware,3,High,-0.200000,0,APPROVE
4,5,My MX Master 3S Mouse broke within one week of purchase. It won't turn on anymore.,my mx master 3s mouse broke within one week of purchase. it won t turn on anymore.,hardware,3,High,0.000000,0,APPROVE
5,6,Got the Logitech MX Master 3S Mouse and the screen was cracked when I opened the package.,got the logitech mx master 3s mouse and the screen was cracked when i opened the package.,screen,3,High,0.000000,0,APPROVE
6,7,"My ROG Strix G15 Laptop is completely dead. No lights, no response, nothing works.","my rog strix g15 laptop is completely dead. no lights, no response, nothing works.",hardware,3,High,-0.200000,0,APPROVE
7,8,The Anker PowerCore 20000 arrived broken. The power button does not respond at all.,the anker powercore ID arrived broken. the power button does not respond at all.,hardware,3,High,-0.400000,0,APPROVE
8,9,Got the Logitech MX Master 3S Mouse and the screen was cracked when I opened the package.,got the logitech mx master 3s mouse and the screen was cracked when i opened the package.,screen,3,High,0.000000,0,APPROVE
9,10,Got the OnePlus Nord CE 3 and the screen was cracked when I opened the package.,got the oneplus nord ce 3 and the screen was cracked when i opened the package.,screen,3,High,0.000000,0,APPROVE
